# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.*, feb.imp_prev
    FROM mar LEFT JOIN feb
      ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL AND feb.imp_prev >= 100
"""
df = conn.execute(momentum_query).df()
content_df = conn.execute(f"""
    SELECT content_hash_id, word_count
    FROM read_parquet('{REL}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()
df = df.merge(content_df, on="content_hash_id", how="inner")
df["is_declining"] = (df["imp_last"] < 0.8 * df["imp_prev"]).astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())

feature_cols = ["avg_position_mar", "imp_prev", "word_count"]
np.random.seed(42)
clients = df["client_hash_id"].unique()
np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = df["client_hash_id"].isin(test_clients)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(
    df[~test_mask][feature_cols].fillna(0), df[~test_mask]["is_declining"]
)
df["model_score"] = rf.predict_proba(df[feature_cols].fillna(0))[:, 1]

def reason_code(row):
    if row["avg_position_mar"] > 20:
        return "deep_position_prior_traffic"
    elif row["avg_position_mar"] > 10:
        return "striking_distance_slipping"
    elif pd.isna(row["avg_position_mar"]):
        return "no_position_data"
    else:
        return "page_one_monitor_only"

def action_label(row):
    if row["model_score"] >= 0.6 and row["avg_position_mar"] > 10:
        return "review_for_refresh"
    elif row["model_score"] >= 0.6:
        return "review_low_priority"
    else:
        return "no_action"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df.apply(action_label, axis=1)

queue = df.sort_values("model_score", ascending=False)
print(queue[["content_hash_id", "avg_position_mar", "imp_prev", "model_score", "reason_code", "action"]].head(10).to_string(index=False))

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         content_hash_id  avg_position_mar  imp_prev  model_score      reason_code              action
content_f504fc1a019390dd               NaN     110.0     0.946916 no_position_data review_low_priority
content_06014c12341ea460               NaN     113.0     0.946916 no_position_data review_low_priority
content_e52d791129006c07               NaN     115.0     0.946916 no_position_data review_low_priority
content_4203003f17e38c5b               NaN     111.0     0.946916 no_position_data review_low_priority
content_787fc1da090116db               NaN     118.0     0.946821 no_position_data review_low_priority
content_a2b4897c296e50a6               NaN     108.0     0.946210 no_position_data review_low_priority
content_4e373c521c5346d9               NaN     108.0     0.946210 no_position_data review_low_priority
content_72171daf1537a376               NaN     106.0     0.944261 no_position_data review_low_priority
content_d1012bc014c546f4               NaN     103.0     0.940165 no_posi

deep_position_prior_traffic — page ranks poorly (position > 20) despite having earned real traffic before; a plausible refresh candidate.
striking_distance_slipping — page sits just off page 1 (position 11-20), FlyRank's own highest-ROI optimization zone; small improvements here are historically the fastest wins.
no_position_data — no reliable position reading this month; flagged separately, never silently treated as "ranked fine."
page_one_monitor_only — already ranks well; the model may still flag it, but action defaults to monitoring, not rewriting a page that isn't actually struggling.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: a prioritization aid for a content editor deciding which pages to review first, within this specific dataset's client roster. Not a page-quality score, not a ranking-algorithm predictor, and not validated on any client outside the ~34 currently in this dataset.

Limits:

This is a cross-sectional, observational model — it shows association, not causation. A page flagged deep_position_prior_traffic is associated with decline in this data; refreshing it is not guaranteed to reverse that.
The client-holdout evaluation used only ~6 test clients per split — Precision@10 varied from 0.40 to 1.00 across random seeds (w05/w06 findings). Any single number from this model carries real uncertainty; treat scores as directional, not precise.
word_count contributes to the model's ranking power but was flagged as a likely client-fingerprint effect (w05), not a genuine content-quality driver — so a high score should not be explained to a stakeholder as "this page needs more words."


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human must check, before acting on any flagged page:

Confirm the page's business context hasn't changed (recent rebrand, product deprecation, planned page removal) — the model has no visibility into this.
Confirm avg_position_mar isn't no_position_data being misread as "fine" — check the reason code, not just the score.
Confirm the client isn't one of the small handful driving most of the model's apparent skill (w05's per-client audit found 2 tiny clients responsible for most of the "perfect" Precision@10 in some splits) — a page's flag is less trustworthy if it comes from an atypical client.

Should NEVER be automated:

Auto-publishing rewritten content without editorial review.
Auto-deleting or de-indexing any page based on this score alone.
Treating model_score as a client-facing "quality grade" — it is an internal triage aid only.
Retraining or re-ranking without re-running the leakage/split-stability checks from w05/w06 first.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [5]:
print("Decline base rate this run:", df["is_declining"].mean().round(3))
print("Number of clients in dataset:", df["client_hash_id"].nunique())

Decline base rate this run: 0.181
Number of clients in dataset: 34


Retrain/re-check triggers:

Base rate drift: if the decline rate moves meaningfully from the 0.18 observed this run, the model's calibration should be re-validated, not assumed to still hold.
New clients onboarded: given how few clients (~34) drove this model, adding even a handful of new clients changes the split composition materially — rerun the multi-seed stability check (w05) before trusting scores on them.
Another bulk-update event: w04's signal audit found a single-day bulk edit contaminating 97% of the "valid" staleness signal — watch for a repeat (a mass CMS migration, bulk republish) that could silently distort content_updated_date-based features again.
Schedule: re-validate quarterly at minimum, or immediately after any known data-pipeline change upstream (BigQuery sync changes, new client onboarding, GA4/GSC integration changes).

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os, json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)  # stays out of git (data file, per CI leak-guard)

metrics = {
    "decline_base_rate": round(df["is_declining"].mean(), 3),
    "n_clients": int(df["client_hash_id"].nunique()),
    "n_pages_scored": len(df),
    "features_used": feature_cols,
    "features_excluded": ["content_age_days (MIXED signal, w04)", "gsc_clicks_mar (label-derived, leakage)"],
    "known_limitation": "word_count in model but likely client-fingerprint effect, not content-quality signal (w05)",
}
with open("work/outputs/action_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Exported queue CSV and metrics JSON.")
print(json.dumps(metrics, indent=2))

Exported queue CSV and metrics JSON.
{
  "decline_base_rate": 0.181,
  "n_clients": 34,
  "n_pages_scored": 76738,
  "features_used": [
    "avg_position_mar_filled",
    "has_position_data",
    "imp_prev",
    "word_count"
  ],
  "features_excluded": [
    "content_age_days (MIXED signal, w04)",
    "gsc_clicks_mar (label-derived, leakage)"
  ],
  "known_limitation": "word_count in model but likely client-fingerprint effect, not content-quality signal (w05)"
}


In [4]:
df["has_position_data"] = df["avg_position_mar"].notna().astype(int)
df["avg_position_mar_filled"] = df["avg_position_mar"].fillna(df["avg_position_mar"].median())  # neutral, not 0

feature_cols = ["avg_position_mar_filled", "has_position_data", "imp_prev", "word_count"]
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(
    df[~test_mask][feature_cols], df[~test_mask]["is_declining"]
)
df["model_score"] = rf.predict_proba(df[feature_cols])[:, 1]


action_priority = {"review_for_refresh": 0, "review_low_priority": 1, "no_action": 2}
queue = df.copy()
queue["reason_code"] = queue.apply(reason_code, axis=1)
queue["action"] = queue.apply(action_label, axis=1)
queue["_sort_key"] = queue["action"].map(action_priority)
queue = queue.sort_values(["_sort_key", "model_score"], ascending=[True, False]).drop(columns="_sort_key")

print(queue[["content_hash_id", "avg_position_mar", "imp_prev", "model_score", "reason_code", "action"]].head(10).to_string(index=False))

         content_hash_id  avg_position_mar  imp_prev  model_score                reason_code             action
content_820a8bd0a2587fb7         10.964770    3719.0     0.761706 striking_distance_slipping review_for_refresh
content_1c90bfc0ba3204ac         10.001145    3734.0     0.761473 striking_distance_slipping review_for_refresh
content_087632fc68dcd6db         10.450337    5058.0     0.761404 striking_distance_slipping review_for_refresh
content_b85c606ed4e64eb2         13.192944   24834.0     0.754633 striking_distance_slipping review_for_refresh
content_895c77b109eea273         11.732036    4308.0     0.750167 striking_distance_slipping review_for_refresh
content_bc15b2472b88cd59         10.532188    6947.0     0.747889 striking_distance_slipping review_for_refresh
content_cf1af462f6317c31         14.505322   30632.0     0.744353 striking_distance_slipping review_for_refresh
content_44f9d2a1ce416af2         10.224632   23641.0     0.741337 striking_distance_slipping review_for_

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.